# Insider Threat Detection — Transformer Encoder (CERT r4.2)

Modularni notebook. Svaka faza poziva module iz `src/`. Prilagodi putanje u `src/config.py`.

## 0. Postavke

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src import config as C, data_loading as D, feature_engineering as FE
from src import sequences as S, training as T, visualization as V
from src.transformer_model import build_transformer
import numpy as np, tensorflow as tf, json
np.random.seed(C.SEED); tf.random.set_seed(C.SEED)
T.enable_mixed_precision()

## 1. EDA

In [ ]:
summary = D.explore_dataset(verbose=True)
ins = D.load_insiders(); print('Insajdera:', ins['user'].nunique())

## 2. Feature engineering

In [ ]:
df = FE.build_daily_features(use_cache=True)
feat_cols = FE.feature_columns(df)
print('Znacajki:', len(feat_cols)); print(D.class_distribution(df))
V.plot_class_distribution(df); df.head()

## 3. Odabir duljine sekvence

In [ ]:
best_L, seq_results = T.select_best_seq_length(df, feat_cols); print('Najbolja:', best_L)

## 4. Sljedovi + podjela + skaliranje

In [ ]:
X, y, groups = S.make_sequences(df, feat_cols, seq_len=best_L)
tr, va, te = S.split_by_user(groups, y)
X, scaler = S.scale_sequences(X, tr, len(feat_cols))
print('X=', X.shape)

## 5. Optuna

In [ ]:
best_params, study = T.optimize_hyperparams(X, y, tr, va, len(feat_cols), best_L); best_params

## 6. Finalno treniranje

In [ ]:
sel = S.subsample_negatives(tr, y); cw = S.compute_class_weight(y[sel])
model = build_transformer(best_L, len(feat_cols), **best_params)
mpath = os.path.join(C.MODELS_DIR,'transformer_best.keras')
hist = model.fit(X[sel], y[sel], validation_data=(X[va], y[va]), epochs=C.MAX_EPOCHS,
                 batch_size=C.BATCH_SIZE, class_weight=cw, callbacks=T.make_callbacks(mpath), verbose=2)
V.plot_learning_curves(hist)

## 7. Evaluacija

In [ ]:
metrics, cm, probs = T.evaluate(model, X[te], y[te]); print(metrics)
V.plot_roc(y[te], probs); V.plot_pr(y[te], probs); V.plot_confusion(cm)

## 8. Interpretabilnost

In [ ]:
imp = T.permutation_importance(model, X[te], y[te], feat_cols, n_repeats=3)
V.plot_feature_importance(imp); list(imp.items())[:10]

In [ ]:
per_step = T.attention_summary(model, X[te][:512])
if per_step is not None: V.plot_attention(per_step)